# Tutorial 07 · Object Detection and Instance Segmentation with YOLO

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_07_Object_Detection_and_Segmentation/tutorial_07_pytorch.ipynb)

Find objects, separate their pixels, inspect confidence and overlap thresholds, and run a small detection fine-tuning workflow with PyTorch-backed YOLO11.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** Python, tensors, and transfer learning (Tutorials 03–06).

**Resources:** CPU supported. Two nano pretrained models total roughly 12 MB; COCO8 is about 1 MB. Default training is two epochs on four images. No paid runtime is required; allow 60–90 minutes for reading and exercises.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals

- Distinguish classification (one image label), detection (boxes and classes), semantic segmentation (pixel classes), and instance segmentation (a separate mask for each object).
- Read pixel `xyxy` boxes, confidence scores, class names, and aligned masks.
- Apply confidence and non-maximum-suppression (NMS) IoU thresholds at inference.
- Inspect YOLO labels, generate a dataset YAML, train, and reload a checkpoint.

We use **YOLO11 nano** consistently, rather than mixing YOLOv3 prose with YOLO11 code. The main example comes with the installed Ultralytics package. Training uses COCO8, a tiny subset of COCO training images: **pretrained weights have already seen the source distribution and may have seen these exact images**. This is a pipeline smoke test, not an independent accuracy benchmark. There is no held-out test claim.

## 1. Environment

Run this before imports in a fresh Colab runtime. We pin the YOLO package to the release used for this tutorial and retain Colab's existing PyTorch stack. Installation occurs inside Colab, not on your computer. A resolver may install missing transitive dependencies; restart and run from the top if you previously imported a different package version.

In [ ]:
import importlib.metadata, importlib.util, subprocess, sys
ULTRALYTICS_VERSION = "8.4.158"
try:
    installed = importlib.metadata.version("ultralytics")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != ULTRALYTICS_VERSION:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",f"ultralytics=={ULTRALYTICS_VERSION}"])
missing = [package for module,package in {"pandas":"pandas","matplotlib":"matplotlib","yaml":"pyyaml"}.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","--quiet",*missing])
print("YOLO package:", importlib.metadata.version("ultralytics"))

In [ ]:
import os, json, time, platform, hashlib, urllib.request, zipfile
from pathlib import Path
os.environ["YOLO_CONFIG_DIR"] = str(Path("data/tutorial_07/config").resolve())
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image
import torch, torchvision, ultralytics, yaml
from torchvision.ops import box_iou, batched_nms
from ultralytics import YOLO, settings
from ultralytics.utils import ASSETS
from IPython.display import display
%matplotlib inline

SEED = 42
DEVICE_PREFERENCE = "auto"  # Choose "cpu" to force CPU.
assert DEVICE_PREFERENCE in {"auto","cpu"}
DEVICE = 0 if DEVICE_PREFERENCE == "auto" and torch.cuda.is_available() else "cpu"
RUN_TRAINING = True  # False skips the training section, but retains detection/segmentation.
TRAIN_EPOCHS = 2
IMAGE_SIZE = 320
CONFIDENCE = 0.35
NMS_IOU = 0.5
DATA_DIR = Path("data/tutorial_07").resolve()
OUTPUT_DIR = Path("outputs/tutorial_07").resolve()
DATA_DIR.mkdir(parents=True,exist_ok=True); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
settings.update({"sync":False,"wandb":False,"comet":False,"mlflow":False,"clearml":False,
                 "datasets_dir":str(DATA_DIR)})
np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
versions = {"python":platform.python_version(),"torch":str(torch.__version__),
    "torchvision":str(torchvision.__version__),"ultralytics":ultralytics.__version__,
    "numpy":np.__version__,"pandas":pd.__version__,"matplotlib":matplotlib.__version__,
    "device":str(DEVICE),"seed":SEED}
display(pd.Series(versions,name="Environment"))
run_started = time.perf_counter()

## 2. Input image and model outputs

Use the package's `bus.jpg` example, so no repository-relative `imgs/test.png` is required. PIL gives **RGB**; OpenCV and Ultralytics `plot()` use **BGR** for NumPy arrays. An unnoticed channel swap makes colors wrong. We use RGB explicitly below.

The model resizes/letterboxes internally. Output boxes are mapped back to original pixel coordinates; do not normalize the PIL input yourself. Scores are model confidence scores, not calibrated probabilities or measured accuracy. The sample has no ground-truth annotations here.

In [ ]:
image_path = Path(ASSETS)/"bus.jpg"
assert image_path.exists(), "The pinned package must include its bus example"
image = Image.open(image_path).convert("RGB")
rgb = np.asarray(image)
height,width = rgb.shape[:2]
image_sha256 = hashlib.sha256(image_path.read_bytes()).hexdigest()
detector = YOLO("yolo11n.pt")
segmenter = YOLO("yolo11n-seg.pt")
common = dict(imgsz=IMAGE_SIZE,device=DEVICE,verbose=False,save=False,max_det=100)
detection = detector.predict(image,conf=CONFIDENCE,iou=NMS_IOU,**common)[0]
# Native-resolution masks avoid stretching letterboxed mask pixels to the image.
segmentation = segmenter.predict(image,conf=CONFIDENCE,iou=NMS_IOU,retina_masks=True,**common)[0]
print("Original RGB shape:",rgb.shape)
print("Detection instances:",len(detection),"Segmentation instances:",len(segmentation))

In [ ]:
COLUMNS = ["instance","class_id","class_name","score","x1","y1","x2","y2"]
def box_table(result):
    rows=[]
    if result.boxes is not None:
        for i,b in enumerate(result.boxes):
            label=int(b.cls.item())
            rows.append([i,label,result.names[label],float(b.conf.item()),*b.xyxy[0].cpu().tolist()])
    return pd.DataFrame(rows,columns=COLUMNS)

def aligned_masks(result):
    h,w = result.orig_shape
    if result.masks is None or len(result)==0:
        return np.zeros((0,h,w),dtype=bool)
    masks = result.masks.data.detach().cpu().numpy()>0.5
    assert masks.shape==(len(result.boxes),h,w), "Use retina_masks=True to retain original geometry"
    return masks

def audit_boxes(result):
    table=box_table(result)
    if len(table):
        a=table[["x1","y1","x2","y2"]].to_numpy()
        assert np.isfinite(a).all() and table.score.between(0,1).all()
        assert (a[:,2:]>=a[:,:2]).all()
        h,w=result.orig_shape
        assert (a>=-1e-3).all() and (a[:,[0,2]]<=w+1e-3).all() and (a[:,[1,3]]<=h+1e-3).all()
    return table

detection_table=audit_boxes(detection)
segmentation_table=audit_boxes(segmentation)
masks=aligned_masks(segmentation)
segmentation_table["mask_pixels"] = masks.sum(axis=(1,2))
segmentation_table["image_fraction"] = masks.mean(axis=(1,2)) if len(masks) else []
display(detection_table.round(3));display(segmentation_table.round(3))
# Explicitly test an empty result; checking `if not results` is insufficient for [empty_result].
empty=segmentation[:0]
assert box_table(empty).empty and aligned_masks(empty).shape==(0,height,width)
print("Coordinate bounds, finite scores, mask alignment, and empty-result handling: passed")

## 3. Original image → boxes → instance masks

Each instance gets a deterministic color, even when two objects share the same class. Mask blending affects only predicted pixels, leaving the background unchanged. Different detection and segmentation models may produce different boxes or instance counts; their indices must not be matched blindly.

In [ ]:
palette=np.asarray(plt.get_cmap("tab10").colors)
def overlay_masks(base,masks,alpha=0.45):
    out=base.astype(np.float32).copy()
    for i,mask in enumerate(masks):
        out[mask]=(1-alpha)*out[mask]+alpha*255*palette[i%len(palette)]
    return np.clip(out,0,255).astype(np.uint8)

def draw_boxes(ax,table):
    for row in table.itertuples():
        color=palette[row.instance%len(palette)]
        ax.add_patch(Rectangle((row.x1,row.y1),row.x2-row.x1,row.y2-row.y1,
                               fill=False,color=color,linewidth=2))
        ax.text(row.x1,max(12,row.y1-4),f"{row.class_name} {row.score:.2f}",fontsize=8,
                color="white",bbox=dict(facecolor=color,alpha=0.85,pad=2))

mask_overlay=overlay_masks(rgb,masks)
union=masks.any(axis=0) if len(masks) else np.zeros((height,width),dtype=bool)
assert np.array_equal(mask_overlay[~union],rgb[~union])
assert np.array_equal(overlay_masks(rgb,aligned_masks(empty)),rgb)
fig,axes=plt.subplots(1,3,figsize=(13,6),layout="constrained")
for ax,im,title in zip(axes,[rgb,rgb,mask_overlay],["Original RGB","Detection: boxes + classes","Instance segmentation: separate masks"]):
    ax.imshow(im);ax.set_title(title,fontsize=11);ax.axis("off")
draw_boxes(axes[1],detection_table);draw_boxes(axes[2],segmentation_table)
fig.savefig(OUTPUT_DIR/"detection_vs_segmentation.png",dpi=140)
plt.show()
Image.fromarray(mask_overlay).save(OUTPUT_DIR/"mask_overlay.png")
# Inspect the actual binary mask rather than just its colored overlay.
fig,axes=plt.subplots(1,3,figsize=(12,5),layout="constrained")
axes[0].imshow(rgb);axes[0].set_title("Original")
if len(masks):
    axes[1].imshow(masks[0],cmap="gray",vmin=0,vmax=1);axes[1].set_title("Instance 0: binary mask")
    isolated=np.where(masks[0][...,None],rgb,0)
    axes[2].imshow(isolated);axes[2].set_title("Pixels selected by that mask")
else:
    for ax in axes[1:]:ax.text(0.5,0.5,"No instances at this threshold",ha="center")
for ax in axes:ax.axis("off")
fig.savefig(OUTPUT_DIR/"binary_mask.png",dpi=140);plt.show()

## 4. Confidence versus IoU: two different controls

Confidence removes low-scoring candidates. NMS suppresses overlapping boxes of the same class: for ordinary YOLO11 NMS, a **lower IoU cutoff suppresses overlaps more aggressively**. Neither cutoff is an evaluation metric. Box IoU is intersection area divided by union area; mask IoU measures pixel sets instead.

A synthetic three-box example makes NMS visible even if changing its threshold happens to leave the bus predictions unchanged. Boxes A and B overlap; C is separate. All three have the same class. `batched_nms` preserves overlapping boxes of different classes by default.

In [ ]:
toy_boxes=torch.tensor([[10.,10.,70.,70.],[20.,20.,80.,80.],[100.,20.,140.,60.]])
toy_scores=torch.tensor([0.9,0.8,0.7]);toy_classes=torch.zeros(3,dtype=torch.long)
iou_ab=float(box_iou(toy_boxes[:1],toy_boxes[1:2])[0,0])
assert abs(iou_ab-2500/4700)<1e-6
kept_low=batched_nms(toy_boxes,toy_scores,toy_classes,0.5).tolist()
kept_high=batched_nms(toy_boxes,toy_scores,toy_classes,0.7).tolist()
assert kept_low==[0,2] and kept_high==[0,1,2]
fig,axes=plt.subplots(1,3,figsize=(12,3.6),layout="constrained")
for ax,indices,title in zip(axes,[list(range(3)),kept_low,kept_high],[f"Candidates; IoU(A,B)={iou_ab:.3f}","NMS IoU=0.5","NMS IoU=0.7"]):
    for i in indices:
        x1,y1,x2,y2=toy_boxes[i].tolist()
        ax.add_patch(Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color=palette[i],lw=3))
        ax.text(x1,y1-3,f"{'ABC'[i]}: {toy_scores[i]:.1f}",color=palette[i])
    ax.set(xlim=(0,155),ylim=(100,0),title=title);ax.set_aspect("equal")
fig.savefig(OUTPUT_DIR/"nms_example.png",dpi=140);plt.show()

In [ ]:
threshold_rows=[]
for conf in [0.2,0.5,0.8]:
    for iou in [0.3,0.7]:
        result=detector.predict(image,conf=conf,iou=iou,**common)[0]
        table=audit_boxes(result)
        threshold_rows.append({"confidence":conf,"nms_iou":iou,"instances":len(table),
                               "classes":", ".join(table.class_name.tolist())})
threshold_table=pd.DataFrame(threshold_rows)
display(threshold_table)
print("Both thresholds were passed to model.predict; counts alone do not measure correctness.")

## 5. Save predictions and verify the export

Export human-readable boxes and machine-readable masks. NPZ preserves the full `(instances, height, width)` mask array; CSV instance indices link rows to masks. No ground truth is available for this sample, so we report counts and geometry checks, not accuracy or mAP. Reloading the exported arrays checks serialization, not model correctness.

In [ ]:
detection_table.to_csv(OUTPUT_DIR/"detections.csv",index=False)
segmentation_table.to_csv(OUTPUT_DIR/"instances.csv",index=False)
threshold_table.to_csv(OUTPUT_DIR/"threshold_experiment.csv",index=False)
np.savez_compressed(OUTPUT_DIR/"instance_masks.npz",masks=masks)
assert np.array_equal(np.load(OUTPUT_DIR/"instance_masks.npz",allow_pickle=False)["masks"],masks)
reloaded=pd.read_csv(OUTPUT_DIR/"detections.csv")
assert len(reloaded)==len(detection_table)
np.testing.assert_allclose(reloaded[["score","x1","y1","x2","y2"]],
                           detection_table[["score","x1","y1","x2","y2"]],rtol=1e-6,atol=1e-6)
print("CSV/NPZ round trips passed.")

## 6. From your labels to a runnable training dataset

A detection label row is `class_id x_center y_center width height`, with coordinates normalized to image size. `class_id` is a zero-based integer; class names and ordering must agree with the model/data YAML. Segmentation training needs polygon annotations instead—box labels are not instance masks.

We download COCO8, audit label ranges and disjoint train/validation image paths and hashes, and generate our own YAML with an absolute dataset path. It contains four training and four validation images. These checks prevent accidental within-experiment overlap but **do not remove overlap with COCO pretraining**. There is no test split. For a real custom dataset, split by acquisition site/video/subject as appropriate before training; keep a separate unseen test set.

In [ ]:
training_report={"executed":False}
DATA_URL="https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip"
if RUN_TRAINING:
    archive_path=DATA_DIR/"coco8.zip"
    if not archive_path.exists():
        temp=archive_path.with_suffix(".part")
        with urllib.request.urlopen(DATA_URL,timeout=120) as response:
            temp.write_bytes(response.read())
        temp.replace(archive_path)
    archive_sha256=hashlib.sha256(archive_path.read_bytes()).hexdigest()
    with zipfile.ZipFile(archive_path) as archive:
        for name in archive.namelist():
            assert (DATA_DIR/name).resolve().is_relative_to(DATA_DIR)
        archive.extractall(DATA_DIR)
    dataset_root=DATA_DIR/"coco8"
    split_images={split:sorted((dataset_root/"images"/split).glob("*.jpg")) for split in ["train","val"]}
    assert all(len(paths)==4 for paths in split_images.values())
    hashes={split:{hashlib.sha256(p.read_bytes()).hexdigest() for p in paths} for split,paths in split_images.items()}
    assert not hashes["train"]&hashes["val"]
    label_rows=[]
    for split,paths in split_images.items():
        for path in paths:
            label_path=dataset_root/"labels"/split/(path.stem+".txt")
            labels=np.loadtxt(label_path,dtype=float,ndmin=2) if label_path.stat().st_size else np.empty((0,5))
            assert labels.shape[1]==5 and np.isfinite(labels).all()
            if len(labels):
                assert ((labels[:,0]>=0)&(labels[:,0]<len(detector.names))&(labels[:,0]==labels[:,0].astype(int))).all()
                assert ((labels[:,1:]>=0)&(labels[:,1:]<=1)).all() and (labels[:,3:]>0).all()
            label_rows.append({"split":split,"image":path.name,"objects":len(labels)})
    config={"path":str(dataset_root),"train":"images/train","val":"images/val","names":detector.names}
    yaml_path=DATA_DIR/"tutorial07_data.yaml"
    yaml_path.write_text(yaml.safe_dump(config,sort_keys=False))
    display(pd.DataFrame(label_rows));print(yaml_path.read_text()[:350])
    # Demonstrate normalized-center to original-pixel conversion on a labelled image.
    sample=split_images["train"][0]
    sample_rgb=np.asarray(Image.open(sample).convert("RGB"));h,w=sample_rgb.shape[:2]
    labels=np.loadtxt(dataset_root/"labels/train"/(sample.stem+".txt"),ndmin=2)
    fig,ax=plt.subplots(figsize=(7,5),layout="constrained");ax.imshow(sample_rgb)
    for cls,cx,cy,bw,bh in labels:
        x1,y1=(cx-bw/2)*w,(cy-bh/2)*h
        ax.add_patch(Rectangle((x1,y1),bw*w,bh*h,fill=False,color="lime",lw=2))
        ax.text(x1,max(y1,12),detector.names[int(cls)],color="black",bbox=dict(facecolor="lime",alpha=0.8))
    ax.set_title("Training annotations (ground truth), not predictions");ax.axis("off")
    fig.savefig(OUTPUT_DIR/"training_annotations.png",dpi=140);plt.show()
else:
    print("Training and dataset download explicitly skipped.")

## 7. Two-epoch detection fine-tuning smoke test

A separate model preserves our original inference example. We specify a small batch, CPU-compatible settings, seed, optimizer, and output directory. Two epochs on four images only demonstrate the mechanics. YOLO reports **mAP50** (AP at IoU 0.50) and **mAP50–95** (averaged over IoU 0.50:0.05:0.95); these are not classification accuracy. On this tiny, potentially pretraining-overlapping validation set they are diagnostic outputs, not credible generalization estimates.

`RUN_TRAINING=False` skips this section. For a real study, use an independent dataset and sufficient training, select settings on validation only, and evaluate the held-out test split once. GPU or a longer training run is a separate experiment.

In [ ]:
if RUN_TRAINING:
    trainer_model=YOLO("yolo11n.pt")
    training_started=time.perf_counter()
    trainer_model.train(data=str(yaml_path),epochs=TRAIN_EPOCHS,imgsz=IMAGE_SIZE,batch=2,
        device=DEVICE,workers=0,seed=SEED,deterministic=True,amp=False,optimizer="SGD",lr0=0.001,
        mosaic=0.0,mixup=0.0,close_mosaic=0,cache=False,plots=False,save=True,val=True,
        project=str(OUTPUT_DIR),name="detection_training",exist_ok=True,verbose=False)
    training_dir=Path(trainer_model.trainer.save_dir)
    checkpoint=training_dir/"weights/best.pt"
    assert checkpoint.is_file()
    history=pd.read_csv(training_dir/"results.csv")
    history.columns=history.columns.str.strip()
    loss_columns=[c for c in history.columns if c.startswith("train/") and c.endswith("loss")]
    assert len(history)==TRAIN_EPOCHS and loss_columns
    assert np.isfinite(history[loss_columns].to_numpy()).all()
    reloaded_model=YOLO(str(checkpoint))
    replay=reloaded_model.predict(image,conf=CONFIDENCE,iou=NMS_IOU,**common)[0]
    audit_boxes(replay)
    assert replay.orig_shape==(height,width) and reloaded_model.names==detector.names
    fig,ax=plt.subplots(figsize=(7,3.5),layout="constrained")
    for column in loss_columns:ax.plot(history.epoch,history[column],marker="o",label=column)
    ax.set(xlabel="Epoch",ylabel="Training loss",title="Two-epoch pipeline smoke test");ax.legend()
    fig.savefig(OUTPUT_DIR/"training_losses.png",dpi=140);plt.show()
    training_report={"executed":True,"epochs":TRAIN_EPOCHS,"train_images":4,"validation_images":4,
        "dataset_url":DATA_URL,"archive_sha256":archive_sha256,"checkpoint":str(checkpoint.relative_to(OUTPUT_DIR)),
        "finite_training_losses":True,"checkpoint_reload_inference":True,
        "last_epoch":{k:float(v) for k,v in history.iloc[-1].items()},
        "seconds":time.perf_counter()-training_started,
        "interpretation":"Pipeline smoke test only; COCO8 is not independent of COCO pretraining; no test estimate."}
    print("Training, finite-loss checks, and reloaded checkpoint inference passed.")

In [ ]:
report={"environment":versions,"image_sha256":image_sha256,"image_shape":list(rgb.shape),
    "models":["yolo11n.pt","yolo11n-seg.pt"],"image_size":IMAGE_SIZE,
    "confidence":CONFIDENCE,"nms_iou":NMS_IOU,"detection_count":len(detection_table),
    "segmentation_count":len(segmentation_table),"mask_shape":list(masks.shape),
    "threshold_experiment":threshold_rows,"training":training_report,
    "checks":{"box_bounds":True,"mask_alignment":True,"empty_result":True,
              "background_unchanged":True,"toy_iou_nms":True,"export_roundtrip":True},
    "seconds_after_imports":time.perf_counter()-run_started}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(report,indent=2,allow_nan=False))
print("TUTORIAL_07_COMPLETE")
print(json.dumps(report,indent=2,allow_nan=False))

## Exercises and interpretation

1. Explain why two instances of the same class need separate masks, and why the detector and segmenter may disagree.
2. Choose a confidence threshold before examining results. Compare 0.2 and 0.8: which visible objects disappear? Counts are not recall without annotations.
3. Change the synthetic NMS classes so boxes A and B are different classes. Predict what class-aware NMS will keep.
4. Try a local image only after the default run succeeds: replace `image_path`, then rerun inference onward. Avoid assuming all object categories are in COCO's 80 classes.
5. Design a genuinely independent custom-data split. Specify label format, leakage prevention, validation selection, test metrics, and data/weight provenance. Do not describe the COCO8 smoke test as evidence that fine-tuning improved performance.

Exports are under `outputs/tutorial_07/`: figures, box CSV files, binary masks, the run report, and (when enabled) training logs and `best.pt`.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_07/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

Adapted and reworked for independent Colab use from [PEESE Tutorial 07](https://github.com/PEESEgroup/SysEn6888/tree/0b326ded95e415d7a60660879aaa737e3c7d79ea/Tutorial_07_Object_Detection_and_Segmentation), Copyright (c) 2025 PEESE. The upstream MIT notice is retained in the repository `LICENSE`. This adaptation does not imply course endorsement.

References: [YOLO11](https://docs.ultralytics.com/models/yolo11/), [prediction arguments and result objects](https://docs.ultralytics.com/modes/predict/), [training](https://docs.ultralytics.com/modes/train/), and [COCO8](https://docs.ultralytics.com/datasets/detect/coco8/).

Ultralytics software, models, and example assets have their own [license terms](https://github.com/ultralytics/ultralytics/blob/main/LICENSE); the upstream course MIT notice does not replace them. COCO images and annotations have separate [terms](https://cocodataset.org/#termsofuse). No image dataset or pretrained weights are redistributed here. The bus image is loaded from the pinned package's bundled examples.

This adaptation fixes local-file dependence, unused IoU controls, RGB/BGR ambiguity, empty-instance handling, and mask-coordinate alignment; adds reproducible exports and a runnable small training workflow.